# DC2 Evaluation Quickstart

This notebook demonstrates the complete DC2 evaluation workflow:

1. **Generate** a sample submission (random noise)
2. **Inspect** the dataset structure
3. **Validate** the format
4. **Run** the evaluation pipeline
5. **Visualise** the results

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-ai-data-challenges/dc2-forecasting-global-ocean-dynamics/blob/main/notebooks/evaluation_quickstart.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ocean-ai-data-challenges/dc2-forecasting-global-ocean-dynamics/main?labpath=notebooks/evaluation_quickstart.ipynb)

## 0. Setup

If running on Colab, install the project first:

```bash
!pip install git+https://github.com/ocean-ai-data-challenges/dc2-forecasting-global-ocean-dynamics.git
```

Otherwise, ensure the `dc2` conda environment is activated (see [Quickstart](https://dc2-forecasting-global-ocean-dynamics.readthedocs.io/en/latest/content/quickstart.html)).

In [ ]:
import json
from pathlib import Path

import numpy as np
import xarray as xr

# Ensure the project root is on the path (works from notebooks/)
import sys
sys.path.insert(0, str(Path.cwd().parent))

## 1. Generate a sample submission

The `create_sample_submission.py` script creates a directory of per-date Zarr stores
filled with random noise that conforms to the DC2 grid specification.

In [ ]:
from scripts.create_sample_submission import create_sample_dataset

OUTPUT_DIR = Path("/tmp/dc2_quickstart_sample")

sample_path = create_sample_dataset(
    OUTPUT_DIR,
    variables=["zos", "thetao", "so", "uo", "vo"],
    seed=42,
)

## 2. Inspect the dataset

Let us open one of the generated Zarr stores and inspect its structure.

In [ ]:
# List generated files
zarr_files = sorted(OUTPUT_DIR.glob("*.zarr"))
print(f"Generated {len(zarr_files)} forecast files:")
for f in zarr_files:
    print(f"  {f.name}")

In [ ]:
# Open the first store
ds = xr.open_zarr(str(zarr_files[0]))
ds

In [ ]:
# Check dimensions
print(f"Latitude:   {ds.lat.values[0]:.2f} to {ds.lat.values[-1]:.2f}  ({len(ds.lat)} points, step {np.diff(ds.lat.values[:2])[0]:.2f})")
print(f"Longitude:  {ds.lon.values[0]:.2f} to {ds.lon.values[-1]:.2f}  ({len(ds.lon)} points, step {np.diff(ds.lon.values[:2])[0]:.2f})")
print(f"Depth:      {ds.depth.values[0]:.1f} m to {ds.depth.values[-1]:.1f} m  ({len(ds.depth)} levels)")
print(f"Time:       {ds.time.values[0]} to {ds.time.values[-1]}  ({len(ds.time)} steps)")
print(f"Variables:  {list(ds.data_vars)}")

## 3. Validate the submission

The `submit.py validate` command checks grid conformity, variable presence, NaN fraction,
and temporal coverage without running the full evaluation.

In [ ]:
import subprocess

result = subprocess.run(
    [
        sys.executable, "dc2/submit.py", "validate",
        str(OUTPUT_DIR),
        "--model-name", "QuickstartSample",
        "--quick",
    ],
    capture_output=True, text=True,
    cwd=str(Path.cwd().parent),
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 4. Inspect the DC2 specification

The `info` subcommand shows the full evaluation configuration.

In [ ]:
result = subprocess.run(
    [sys.executable, "dc2/submit.py", "info", "--config", "dc2"],
    capture_output=True, text=True,
    cwd=str(Path.cwd().parent),
)
print(result.stdout)

## 5. Visualise existing results

If a full evaluation has already been run (e.g. GloNet baseline), we can load and
visualise the results.

In [ ]:
import matplotlib.pyplot as plt

# Load existing GloNet results if available
results_path = Path.cwd().parent / "dc2_output" / "results" / "results_glonet.json"
if not results_path.exists():
    results_path = Path.cwd().parent / "dc2" / "leaderboard_results" / "results_glonet.json"

if results_path.exists():
    with open(results_path) as f:
        data = json.load(f)
    print(f"Loaded results for model: {data['dataset']}")
else:
    print(f"No results file found. Run the full evaluation first:")
    print(f"  python dc2/submit.py run {OUTPUT_DIR} --model-name QuickstartSample")

In [ ]:
# Parse RMSD scores from GLORYS reference, grouped by variable
if results_path.exists():
    model_name = data["dataset"]
    records = data["results"][model_name]

    # Gather GLORYS RMSD results
    glorys_rmsd = {}
    for record in records:
        if record["ref_alias"] != "glorys":
            continue
        lead_time = record.get("lead_time")
        for metric in record["result"]:
            if metric["Metric"] != "rmsd":
                continue
            var_name = metric["Variable"]
            glorys_rmsd.setdefault(var_name, {"lead_times": [], "values": []})
            glorys_rmsd[var_name]["lead_times"].append(lead_time)
            glorys_rmsd[var_name]["values"].append(metric["Value"])

    # Plot RMSD vs lead time for surface variables
    surface_vars = [v for v in glorys_rmsd if "Surface" in v]
    if surface_vars:
        fig, ax = plt.subplots(figsize=(10, 5))
        for var_name in sorted(surface_vars):
            entry = glorys_rmsd[var_name]
            order = np.argsort(entry["lead_times"])
            ax.plot(
                np.array(entry["lead_times"])[order],
                np.array(entry["values"])[order],
                marker="o", label=var_name,
            )
        ax.set_xlabel("Lead time (days)")
        ax.set_ylabel("RMSD")
        ax.set_title(f"{model_name} - Surface RMSD vs GLORYS12")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
# Depth profile: RMSD at different depths (averaged over lead times)
if results_path.exists():
    depth_labels = ["Surface", "50m", "200m", "550m"]

    for phys_var in ["temperature", "salinity"]:
        means = []
        for dl in depth_labels:
            key = f"{dl} {phys_var}"
            if key in glorys_rmsd:
                means.append(np.mean(glorys_rmsd[key]["values"]))
            else:
                means.append(np.nan)

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.barh(depth_labels[::-1], means[::-1])
        ax.set_xlabel("Mean RMSD")
        ax.set_title(f"{model_name} - {phys_var.capitalize()} RMSD by depth")
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        plt.show()

## Next steps

- Replace the random-noise sample with your model predictions
- Run the full evaluation: `python dc2/submit.py run <your_data> --model-name <NAME>`
- Submit results for the [leaderboard](https://dc2-forecasting-global-ocean-dynamics.readthedocs.io/en/latest/content/submission.html#participating-in-the-public-leaderboard)

See the [full documentation](https://dc2-forecasting-global-ocean-dynamics.readthedocs.io/) for details.